# NBM-SPAM: Learned unary bases with low-rank polynomial heads

NBM-SPAM learns unary NBM score channels, uses one segment linearly, and sends the remaining segments through degree-specific SPAM heads.


## Model


Let $z_j^{(q)}(x_j)$ be unary NBM scores reserved for degree $q$. Then

$$
\eta(x)=\beta_0+\sum_j a_jz_j^{(1)}(x_j)
+\sum_{q=2}^{Q}\sum_{r=1}^{R_q}
\alpha_{qr}\left(\sum_jw_{qrj}z_j^{(q)}(x_j)\right)^q.
$$

Higher-order structure is created by SPAM; the NBM stage remains unary.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_preprocessing` and `categorical_preprocessing` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import NBMSPAMClassifier, NBMSPAMLSS, NBMSPAMRegressor


model = NBMSPAMRegressor(
    num_bases=16,
    layer_sizes=[32, 16],
    ranks=[12],
    num_subnets=1,
    featurizer="conv1d",
    output_penalty=1e-4,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

`ranks` sets the SPAM head rank for each polynomial degree. `num_subnets` controls replicated unary basis score channels per polynomial.


In [ ]:
if RUN_TRAINING:
    display({key: model.get_params(deep=False)[key] for key in ("ranks", "num_subnets", "featurizer")})
    display(model.predict_components(X_test).terms.keys())


## Task variants and limits

NBM-SPAM provides regressor, classifier, and LSS estimators with the same hybrid decomposition.
